# PyADM1ODE: Two-Stage Plant Example

This notebook demonstrates a two-stage biogas plant with energy integration (CHP and heating) using the ADM1 (SIMBA# biogas) model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE/blob/master/examples/colab_02_complex_plant.ipynb)

## 1. Setup environment

PyADM1 is pure Python — no Mono / .NET runtime needed.

In [ ]:
!git clone https://github.com/dgaida/PyADM1ODE.git
%cd PyADM1ODE
!pip install -e .

## 2. Import libraries

In [ ]:
from pyadm1 import BiogasPlant, Feedstock
from pyadm1.configurator.plant_configurator import PlantConfigurator

## 3. Configure and run simulation

In [ ]:
# 1. Feedstock
feedstock = Feedstock(
    ["maize_silage_milk_ripeness", "swine_manure"],
    feeding_freq=24,
    total_simtime=10,
)
plant = BiogasPlant("Two-Stage Plant")
cfg = PlantConfigurator(plant, feedstock)

# 2. Stage 1 — Hydrolysis pre-tank, 45 deg C, short HRT
cfg.add_digester(
    digester_id="digester_1",
    V_liq=500.0,
    V_gas=75.0,
    T_ad=318.15,
    Q_substrates=[11.4, 6.1],
)

# 3. Stage 2 — Methanogenesis, 35 deg C
cfg.add_digester(
    digester_id="digester_2",
    V_liq=1000.0,
    V_gas=150.0,
    T_ad=308.15,
    Q_substrates=[0.0, 0.0],
)

# 4. CHP and heating
cfg.add_chp("chp_1", P_el_nom=500.0)
cfg.add_heating("heating_1", target_temperature=318.15)
cfg.add_heating("heating_2", target_temperature=308.15)

# 5. Connect components
cfg.connect("digester_1", "digester_2", "liquid")
cfg.auto_connect_digester_to_chp("digester_1", "chp_1")
cfg.auto_connect_digester_to_chp("digester_2", "chp_1")
cfg.auto_connect_chp_to_heating("chp_1", "heating_1")
cfg.auto_connect_chp_to_heating("chp_1", "heating_2")

# 6. Initialize and simulate
plant.initialize()
results = plant.simulate(duration=5.0, dt=1.0, save_interval=1.0)

print("\nSimulation completed successfully!")

## 4. Analyze results

In [ ]:
final = results[-1]["components"]
print(f"Total Biogas:  {final['digester_1']['Q_gas'] + final['digester_2']['Q_gas']:.1f} m^3/d")
print(f"Total Methane: {final['digester_1']['Q_ch4'] + final['digester_2']['Q_ch4']:.1f} m^3/d")
print(f"CHP Power:     {final['chp_1'].get('P_el', 0):.1f} kW")